# v4 dash-cam pothole model — beat 0.182 on the Detroit exam (Ayaan's brief, chat 01:04 UTC)

**Bar:** shipped dash-cam model = v3+v1 weight blend 70/30, Detroit exam box F1 **0.182** (v3 0.157, v1 0.103).
**v4 data** (`pipeline/autolabel/build_v4_data.py`): Detroit train + road-band full-res tiles + v3's mistakes ×3
(mostly tar snakes / lane edges) + pasted small potholes, ~50% "nothing here". Split by video, test never touched.
**Train** from v3 at 640. **Then** blend v4 with v1 (70/30 and 85/15) and score everything — including the shipped
blend as a sanity baseline — on the same 215-frame exam.

**Setup:** A100 (or L4) · `Runtime → Run all` · ~40–50 min. Last cell prints `report.md` and saves the best
ONNX to `still-there-potholes/handoff_v4/`. **Ship rule:** best box F1 wins, must beat 0.182.

## 1 · GPU, Drive, repo, v3 weights

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU"
!pip -q install ultralytics onnx onnxruntime onnxslim
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, glob
DRIVE_DIR = '/content/drive/MyDrive/still-there-potholes'
if os.path.isdir('/content/repo/.git'):
    !git -C /content/repo pull -q
else:
    !git clone -q https://github.com/ayaanamir/RoadMapWayneAWSHackathon /content/repo
V3 = next(p for p in (f'{DRIVE_DIR}/handoff_detroit/best.pt', f'{DRIVE_DIR}/runs/y11n_dashcam_detroit/weights/best.pt') if os.path.exists(p))
shutil.copy(V3, '/content/v3_best.pt')
V1_ONNX = '/content/repo/pipeline/train/results/v1_phone/pothole.onnx'
SHIPPED = '/content/repo/apps/live/model/pothole-dashcam.onnx'   # v3+v1 blend, exam 0.182
for p in ('/content/v3_best.pt', V1_ONNX, SHIPPED): print(os.path.exists(p), p)

## 2 · Detroit data → v4 training set

In [ ]:
!python3 /content/repo/pipeline/autolabel/pull_reviewed.py --api https://wayneawshackathon.vercel.app --out /content/detroit
!python3 /content/repo/pipeline/autolabel/build_v4_data.py --images /content/detroit/train/images --labels /content/detroit/train/labels --model /content/v3_best.pt --conf 0.35 --out /content/ds_v4
print(open('/content/ds_v4/data.yaml').read())
for s in ('train', 'val', 'valid', 'test'):
    d = f'/content/ds_v4/{s}/images'
    if os.path.isdir(d):
        labs = glob.glob(f'/content/ds_v4/{s}/labels/*.txt')
        print(s, len(os.listdir(d)), 'images,', sum(os.path.getsize(l) == 0 for l in labs), 'empty labels')

## 3 · Train v4 from v3 (≈20–30 min on A100) — resumes after a disconnect

In [ ]:
from ultralytics import YOLO
RUN = f'{DRIVE_DIR}/runs/v4_detroit'
last, done = f'{RUN}/weights/last.pt', f'{RUN}/weights/TRAINING_DONE'
if os.path.exists(done):
    print('Already trained — skipping. Delete', done, 'to train again.')
elif os.path.exists(last):
    print('Resuming', last); YOLO(last).train(resume=True)
else:
    YOLO('/content/v3_best.pt').train(data='/content/ds_v4/data.yaml', imgsz=640, epochs=40, patience=12,
        lr0=0.002, warmup_epochs=1, hsv_v=0.5, hsv_s=0.7, close_mosaic=10, batch=-1, cache='ram',
        workers=min(8, os.cpu_count() or 2), project=f'{DRIVE_DIR}/runs', name='v4_detroit', exist_ok=True)
open(done, 'w').close()
V4_PT = f'{RUN}/weights/best.pt'

## 4 · Export v4 + blend with v1 (70/30, 85/15) — the trick that took v3 0.157 → 0.182

In [ ]:
import onnx, numpy as np
from onnx import numpy_helper
V4 = YOLO(V4_PT).export(format='onnx', imgsz=640)
def blend(a_path, b_path, a, out):
    A = onnx.load(a_path); B = {i.name: numpy_helper.to_array(i) for i in onnx.load(b_path).graph.initializer}
    n = 0
    for k, init in enumerate(A.graph.initializer):
        w = numpy_helper.to_array(init)
        if w.dtype == np.float32 and init.name in B and B[init.name].shape == w.shape:
            A.graph.initializer[k].CopyFrom(numpy_helper.from_array((a*w + (1-a)*B[init.name]).astype(np.float32), init.name)); n += 1
    onnx.save(A, out); print(f'{os.path.basename(out)}: blended {n}/{len(A.graph.initializer)} tensors')
for a in (0.7, 0.85): blend(V4, V1_ONNX, a, f'/content/v4v1_{int(a*100)}.onnx')

## 5 · The Detroit exam — v4, both blends, and the shipped blend (0.182) as baseline

In [ ]:
CANDS = {'v4': V4, 'v4v1_70': '/content/v4v1_70.onnx', 'v4v1_85': '/content/v4v1_85.onnx', 'shipped_v3v1_70': SHIPPED}
named = {}
for name, p in CANDS.items():                      # distinct file names so the report rows are readable
    dst = f'/content/exam_models/{name}{os.path.splitext(p)[1]}'; os.makedirs('/content/exam_models', exist_ok=True)
    shutil.copy(p, dst); named[name] = dst
args = ' '.join(f'--model {p}' for p in named.values())
!python3 /content/repo/pipeline/autolabel/eval_models.py --images /content/detroit/test/images --labels /content/detroit/test/labels --out /content/eval_v4 --modes 640 --device 0 {args}
REPORT = open('/content/eval_v4/report.md').read(); print(REPORT)
os.makedirs(f'{DRIVE_DIR}/handoff_v4', exist_ok=True)
open(f'{DRIVE_DIR}/handoff_v4/v4_exam_report.md', 'w').write(REPORT)

## 6 · Save the candidates to Drive (`handoff_v4/`), then disconnect

Ship only the winner, and only if it beats **0.182**: post the report + its Drive link in the agent chat;
Ayaan puts it at `apps/live/model/pothole-dashcam.onnx` (one PR, one deploy). Don't commit .pt/.onnx yourself.

In [ ]:
for name in ('v4', 'v4v1_70', 'v4v1_85'):
    shutil.copy(named[name], f'{DRIVE_DIR}/handoff_v4/{name}.onnx')
shutil.copy(V4_PT, f'{DRIVE_DIR}/handoff_v4/v4_best.pt')
for f in sorted(os.listdir(f'{DRIVE_DIR}/handoff_v4')): print(f'{f:22s} {os.path.getsize(f"{DRIVE_DIR}/handoff_v4/{f}")/1e6:.1f} MB')
print('\nDONE — Runtime → Disconnect and delete runtime to stop burning units.')